# Chapter 14 — Excel Spreadsheets

Source: [Automate the Boring Stuff with Python, 3rd Ed. — Chapter 14](https://automatetheboringstuff.com/3e/chapter14.html)

**Setup note:** Needs `pip install openpyxl`.

This notebook favors *reading* over *writing*: each section has a short explanation followed by real, runnable code. Read it, run it, tweak a value and re-run it — that's the fastest way to build reading fluency.

---


## 14.1 Opening a workbook and reading cells

`openpyxl.load_workbook(path)` opens an `.xlsx` file. A workbook contains sheets (`wb['Sheet1']` or `wb.active`), and a sheet's cells are accessed like `sheet['A1']` or `sheet.cell(row=1, column=1)`.


In [1]:
import openpyxl

wb = openpyxl.Workbook()
sheet = wb.active
sheet['A1'] = 'Name'
sheet['B1'] = 'Age'
sheet['A2'] = 'Alice'
sheet['B2'] = 30
wb.save('demo.xlsx')

wb2 = openpyxl.load_workbook('demo.xlsx')
sheet2 = wb2.active
print(sheet2['A2'].value, sheet2.cell(row=2, column=2).value)


Alice 30


## 14.2 Looping over rows and columns

`sheet.iter_rows()` and `sheet.iter_columns()` (or simply looping over `sheet.rows`) let you process a whole sheet without hardcoding cell addresses — the normal way to read real spreadsheet data.


In [2]:
import openpyxl
wb = openpyxl.load_workbook('demo.xlsx')
sheet = wb.active

for row in sheet.iter_rows(min_row=1, max_row=2, values_only=True):
    print(row)


('Name', 'Age')
('Alice', 30)


## 14.3 Writing and saving workbooks

Setting `sheet['A1'] = value` (or `sheet.cell(row, column).value = value`) writes data; `wb.save(path)` writes the changes back to disk. `wb.create_sheet('Name')` adds a new sheet.


In [3]:
import openpyxl
wb = openpyxl.load_workbook('demo.xlsx')
sheet = wb.active
sheet['C1'] = 'City'
sheet['C2'] = 'Springfield'
wb.save('demo.xlsx')

wb2 = openpyxl.load_workbook('demo.xlsx')
print(wb2.active['C2'].value)

import os
os.remove('demo.xlsx')  # clean up demo file


Springfield


## 14.4 Cell styles and formulas

openpyxl can set fonts, fills, and number formats on cells, and can write Excel formulas as strings (Excel evaluates them when the file is opened — openpyxl itself doesn't calculate formula results).


In [4]:
import openpyxl
from openpyxl.styles import Font

wb = openpyxl.Workbook()
sheet = wb.active
sheet['A1'] = 'Total'
sheet['A1'].font = Font(bold=True, size=14)
sheet['B1'] = '=SUM(1,2,3)'   # Excel computes this when opened, openpyxl just stores the text
wb.save('styled_demo.xlsx')
print('saved styled_demo.xlsx')

import os
os.remove('styled_demo.xlsx')


saved styled_demo.xlsx


## 14.5 Charts

`openpyxl` can add real Excel charts, not just cell values: `openpyxl.chart.BarChart()` (or `LineChart`, `PieChart`, `ScatterChart`) creates a chart object, `openpyxl.chart.Reference(ws, min_col=.., min_row=.., max_row=..)` describes which cell range feeds it data, `chart.add_data(ref)` links them, and `ws.add_chart(chart, 'D2')` places it on the sheet at a given anchor cell. The chart is a real object saved inside the `.xlsx` file — opening it in Excel/Google Sheets/LibreOffice shows an actual rendered chart, not just numbers.

In [5]:
import openpyxl
from openpyxl.chart import BarChart, Reference

wb = openpyxl.Workbook()
ws = wb.active
ws.append(['Month', 'Sales'])
for month, sales in [('Jan', 120), ('Feb', 150), ('Mar', 90), ('Apr', 200)]:
    ws.append([month, sales])

chart = BarChart()
chart.title = 'Monthly Sales'
data = Reference(ws, min_col=2, min_row=1, max_row=5)     # includes the header row as the series title
categories = Reference(ws, min_col=1, min_row=2, max_row=5)
chart.add_data(data, titles_from_data=True)
chart.set_categories(categories)
ws.add_chart(chart, 'D2')

wb.save('sales_chart_demo.xlsx')

# Prove the chart actually persisted inside the saved file
wb2 = openpyxl.load_workbook('sales_chart_demo.xlsx')
ws2 = wb2.active
print('Charts found after reloading the saved file:', len(ws2._charts))
print('Chart title:', ws2._charts[0].title.tx.rich.p[0].r[0].t)

import os
os.remove('sales_chart_demo.xlsx')


Charts found after reloading the saved file: 1
Chart title: Monthly Sales
